# NI GP Prescribing Explorer

Explore prescribing of different medicines by GP practice or HSC Trust area, per capita.

**Data sources** (Open Government Licence):
- [GP Prescribing Data](https://www.opendatani.gov.uk/dataset/gp-prescribing-data) — monthly, from April 2013
- [GP Practice List Sizes](https://www.opendatani.gov.uk/dataset/gp-practice-list-sizes) — quarterly

Published by the Business Services Organisation (BSO) via OpenDataNI.

## 1. Download the data

Run the cells below to fetch the latest data from OpenDataNI. You only need to do this once (files are cached in `data/`).

In [ ]:
# Download practice list sizes and the 3 most recent months of prescribing data.
# Adjust --latest or --year/--month to fetch more or different periods.
# For a full year: !python download_data.py --prescribing --year 2024 --practices

!python download_data.py --practices
!python download_data.py --prescribing --latest 3

## 2. Load and merge the data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from analyse import (
    load_practice_list,
    load_prescribing,
    merge_with_practice_list,
    prescribing_per_capita_by_practice,
    prescribing_per_capita_by_trust,
    top_medicines_by_items,
    top_medicines_by_cost,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

In [ ]:
# Load practice list (most recent quarter available)
practices = load_practice_list()
print(f"{len(practices)} practices loaded")
print(f"Total registered patients: {practices['RegisteredPatients'].sum():,}")
practices.head()

In [ ]:
# Patients per Trust
trust_pop = practices.groupby("Trust")["RegisteredPatients"].sum().sort_values(ascending=False)
trust_pop

In [ ]:
# Load prescribing data (all downloaded files)
rx = load_prescribing()
print(f"{len(rx):,} prescribing rows loaded")
rx.head()

In [ ]:
# Merge prescribing with practice data
merged = merge_with_practice_list(rx, practices)
print(f"{len(merged):,} merged rows")
merged.head()

## 3. What are the most prescribed medicines?

In [ ]:
top_items = top_medicines_by_items(rx, n=20)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=top_items, y="VTM_NM", x="TotalItems", ax=ax)
ax.set_title("Top 20 medicines by total items prescribed")
ax.set_xlabel("Total items")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
top_cost = top_medicines_by_cost(rx, n=20)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=top_cost, y="VTM_NM", x="ActualCost", ax=ax)
ax.set_title("Top 20 medicines by total actual cost")
ax.set_xlabel("Total actual cost (£)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 4. Per-capita prescribing by HSC Trust

Compare prescribing rates across the 5 Health & Social Care Trust areas.

In [ ]:
# All prescribing per capita by trust
trust_summary = prescribing_per_capita_by_trust(merged)
trust_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=trust_summary, x="Trust", y="ItemsPerCapita", ax=axes[0])
axes[0].set_title("Prescribing items per capita by Trust")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(data=trust_summary, x="Trust", y="CostPerCapita", ax=axes[1])
axes[1].set_title("Prescribing cost per capita by Trust (£)")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 5. Explore a specific medicine

Change the `MEDICINE` variable below to explore any drug. Uses substring matching on VTM_NM (substance name).

Examples: `"Amoxicillin"`, `"Omeprazole"`, `"Atorvastatin"`, `"Paracetamol"`, `"Metformin"`, `"Salbutamol"`

In [ ]:
MEDICINE = "Amoxicillin"  # <-- change this to explore different medicines

In [ ]:
# Per-capita by Trust for this medicine
trust_med = prescribing_per_capita_by_trust(merged, medicine_filter=MEDICINE)
print(f"\n{MEDICINE} — per capita by Trust:")
trust_med

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=trust_med, x="Trust", y="ItemsPerCapita", ax=axes[0])
axes[0].set_title(f"{MEDICINE} — items per capita by Trust")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=30)

sns.barplot(data=trust_med, x="Trust", y="CostPerCapita", ax=axes[1])
axes[1].set_title(f"{MEDICINE} — cost per capita by Trust (£)")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Per-capita by practice for this medicine (top 20 and bottom 20)
practice_med = prescribing_per_capita_by_practice(merged, medicine_filter=MEDICINE)

print(f"\n{MEDICINE} — Top 20 practices by items per capita:")
practice_med.head(20)

In [ ]:
print(f"\n{MEDICINE} — Bottom 20 practices by items per capita:")
practice_med.tail(20)

In [ ]:
# Distribution of per-capita prescribing across all practices, coloured by Trust
fig, ax = plt.subplots(figsize=(12, 6))
for trust, group in practice_med.groupby("Trust"):
    ax.scatter(group["RegisteredPatients"], group["ItemsPerCapita"],
               label=trust, alpha=0.6, s=40)

ax.set_xlabel("Practice size (registered patients)")
ax.set_ylabel(f"{MEDICINE} items per capita")
ax.set_title(f"{MEDICINE} prescribing rate vs practice size")
ax.legend(title="Trust", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 6. Compare multiple medicines across Trusts

In [ ]:
MEDICINES_TO_COMPARE = ["Amoxicillin", "Omeprazole", "Atorvastatin", "Paracetamol", "Metformin"]

comparison = []
for med in MEDICINES_TO_COMPARE:
    df = prescribing_per_capita_by_trust(merged, medicine_filter=med)
    df["Medicine"] = med
    comparison.append(df)

comparison_df = pd.concat(comparison, ignore_index=True)

fig, ax = plt.subplots(figsize=(14, 7))
sns.barplot(data=comparison_df, x="Trust", y="ItemsPerCapita", hue="Medicine", ax=ax)
ax.set_title("Per-capita prescribing items by Trust — selected medicines")
ax.set_xlabel("")
ax.set_ylabel("Items per capita")
ax.tick_params(axis="x", rotation=30)
ax.legend(title="Medicine", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 7. Explore by BNF Chapter

The BNF (British National Formulary) organises medicines into chapters:

| Ch | Category |
|----|----------|
| 1  | Gastro-intestinal |
| 2  | Cardiovascular |
| 3  | Respiratory |
| 4  | Central nervous system |
| 5  | Infections |
| 6  | Endocrine |
| 7  | Obstetrics, gynaecology, urinary |
| 8  | Malignant disease, immunosuppression |
| 9  | Nutrition, blood |
| 10 | Musculoskeletal, joint |
| 11 | Eye |
| 12 | Ear, nose, oropharynx |
| 13 | Skin |
| 14 | Immunological, vaccines |
| 15 | Anaesthesia |

In [ ]:
BNF_CHAPTER = "4"  # Central nervous system — change to explore others

# Filter the already-loaded prescribing data
if "BNFChapter" in rx.columns:
    chapter_rx = rx[rx["BNFChapter"].astype(str).str.strip() == BNF_CHAPTER]
    chapter_merged = merge_with_practice_list(chapter_rx, practices)

    chapter_trust = prescribing_per_capita_by_trust(chapter_merged)
    print(f"BNF Chapter {BNF_CHAPTER} — per capita by Trust:")
    display(chapter_trust)

    # Top medicines in this chapter
    top_in_chapter = top_medicines_by_items(chapter_rx, n=15)
    print(f"\nTop 15 medicines in BNF Chapter {BNF_CHAPTER}:")
    display(top_in_chapter)
else:
    print("BNFChapter column not found in the data — check your downloaded files")

## 8. Practice-level variation

Look at how prescribing varies across all practices for a given medicine.

In [ ]:
MEDICINE_VAR = "Omeprazole"  # <-- change this

prac_var = prescribing_per_capita_by_practice(merged, medicine_filter=MEDICINE_VAR)
prac_var = prac_var[prac_var["RegisteredPatients"] > 500]  # exclude very small practices

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of per-capita rates
axes[0].hist(prac_var["ItemsPerCapita"], bins=30, edgecolor="white")
axes[0].set_xlabel(f"{MEDICINE_VAR} items per capita")
axes[0].set_ylabel("Number of practices")
axes[0].set_title(f"Distribution of {MEDICINE_VAR} prescribing rate")
axes[0].axvline(prac_var["ItemsPerCapita"].median(), color="red",
                linestyle="--", label=f"Median: {prac_var['ItemsPerCapita'].median():.3f}")
axes[0].legend()

# Box plot by Trust
sns.boxplot(data=prac_var, x="Trust", y="ItemsPerCapita", ax=axes[1])
axes[1].set_title(f"{MEDICINE_VAR} prescribing rate by Trust")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

# Summary stats
print(f"\n{MEDICINE_VAR} items per capita — summary by Trust:")
prac_var.groupby("Trust")["ItemsPerCapita"].describe()

## 9. Quick search

Search for any medicine by name to see if it appears in the data.

In [ ]:
SEARCH = "statin"  # <-- change this to search

matches = rx[rx["VTM_NM"].str.contains(SEARCH, case=False, na=False)]
unique_medicines = matches["VTM_NM"].unique()
print(f"Found {len(unique_medicines)} matching medicines:")
for m in sorted(unique_medicines):
    items = matches[matches["VTM_NM"] == m]["TotalItems"].sum()
    print(f"  {m}: {items:,.0f} items")